In [2]:
## Interpolation Performance Evaluation

"""
Weak-label validation + interpolation evaluation for gridded predictor surfaces
===============================================================================

This script does two different things on purpose:

1) Continuous-variable interpolation QA/QC
   - temp_air, temp_wet, temp_dew, rh
   - sampled at observed points/times
   - metrics: RMSE, MAE, bias, correlation

2) MRoS weak-label validation
   - mros_plp_proxy is treated as a weak-label phase index,
     NOT as true physical PLP
   - evaluate both:
       a) continuous-index agreement
       b) phase recovery after thresholding
   - inspect thermodynamic distributions using raw MRoS observations
     and interpolated/sampled weak labels

Important interpretation
------------------------
This script is primarily an *internal consistency / weak-label defensibility*
workflow, not a true out-of-sample interpolation skill assessment.

Recommended use
---------------
- Run first on the kriging surface NetCDF
- Then run the same script on the IDW surface NetCDF
- Compare outputs side-by-side

Author: Emma Golub
Date: March 2026
"""

from __future__ import annotations

import warnings
warnings.filterwarnings("ignore")

from pathlib import Path
from typing import Iterable, Optional, Tuple, Dict, List

import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt
import seaborn as sns
import rasterio as rio

from pyproj import Transformer
from rasterio.warp import calculate_default_transform, reproject, Resampling
from scipy.spatial import cKDTree
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    confusion_matrix,
    balanced_accuracy_score,
    precision_recall_fscore_support,
)
from sklearn.metrics import r2_score


# =============================================================================
# 0. CONFIG
# =============================================================================

BASE_DIR = Path().resolve().parent

CONFIG = {
    # ---- Input data ----
    "station_hourly_parquet": BASE_DIR / "outputs/hourly_pipeline/hourly_data/stations_hourly.parquet",
    "mros_hourly_parquet":    BASE_DIR / "outputs/hourly_pipeline/hourly_data/mros_hourly.parquet",
    "dem_path":               BASE_DIR / "Data/elevation/DEM_1km_clipped.tif",

    # ---- NetCDF to evaluate ----
    # "surface_nc": Path("C:/Users/EmmaGolub/Desktop/MRoS_local/local_data/kriging/hourly_predictors_1km_kriging_v1_final.nc"),
    "surface_nc": Path("C:/Users/EmmaGolub/Desktop/MRoS_local/local_data/IDW/hourly_predictors_1km_IDW_full.nc"),

    # ---- Output folder ----
    "out_dir": BASE_DIR / "outputs/hourly_pipeline/evaluation/weak_label_validation_IDW",

    # ---- Time window ----
    "start_time": "2024-10-01 00:00",
    "end_time":   "2025-05-31 23:00",

    # ---- CRS fallback if DEM is geographic ----
    "proj_fallback": "EPSG:26911",

    # ---- MRoS threshold defaults ----
    # Current project thresholds for proxy -> class
    "mros_snow_upper": 20.0,
    "mros_rain_lower": 80.0,

    # ---- Threshold sensitivity grid ----
    "snow_upper_grid": [10, 15, 20, 25, 30, 35, 40],
    "rain_lower_grid": [60, 65, 70, 75, 80, 85, 90],

    # ---- Temperature bins for phase-fraction plots ----
    "temp_bin_width": 0.5,
    "temp_bin_min": -6.0,
    "temp_bin_max":  6.0,
}

OUT_DIR = Path(CONFIG["out_dir"])
OUT_DIR.mkdir(parents=True, exist_ok=True)

sns.set_style("whitegrid")
plt.rcParams["figure.dpi"] = 130


# =============================================================================
# 1. BASIC HELPERS
# =============================================================================

def ensure_tz_naive_hour(s: pd.Series) -> pd.Series:
    """Convert datetime-like series to UTC, floor to hour, then drop timezone."""
    return pd.to_datetime(s, utc=True, errors="coerce").dt.floor("h").dt.tz_localize(None)


def rmse(y_true: np.ndarray, y_pred: np.ndarray) -> float:
    return float(np.sqrt(mean_squared_error(y_true, y_pred)))


def safe_corr(y_true: np.ndarray, y_pred: np.ndarray) -> float:
    if len(y_true) < 2:
        return np.nan
    if np.nanstd(y_true) == 0 or np.nanstd(y_pred) == 0:
        return np.nan
    return float(np.corrcoef(y_true, y_pred)[0, 1])


def classify_mros_proxy(values: np.ndarray,
                        snow_upper: float = 20.0,
                        rain_lower: float = 80.0) -> np.ndarray:
    """
    Convert MRoS phase proxy into hard classes using thresholds.
    Classes:
      <= snow_upper  -> snow
      >= rain_lower  -> rain
      else           -> mix
    """
    values = np.asarray(values, dtype=float)
    out = np.full(values.shape, None, dtype=object)
    out[values <= snow_upper] = "snow"
    out[values >= rain_lower] = "rain"
    mid = (values > snow_upper) & (values < rain_lower)
    out[mid] = "mix"
    return out


def ordered_phase_labels() -> List[str]:
    return ["snow", "mix", "rain"]


def make_temp_bins(bin_min=-6.0, bin_max=6.0, width=0.5) -> np.ndarray:
    return np.arange(bin_min, bin_max + width, width)


# =============================================================================
# 2. DEM / CRS HELPERS
# =============================================================================

def load_dem_and_projected_profile(dem_path: Path,
                                   proj_fallback: str = "EPSG:26911") -> Tuple[np.ndarray, dict, object]:
    """
    Load DEM. If DEM CRS is geographic, reproject in-memory to a projected CRS.
    Returns:
      dem_data, dem_profile, proj_crs
    """
    with rio.open(dem_path) as src:
        dem_crs = src.crs

        if (dem_crs is None) or (not dem_crs.is_projected):
            print(f"DEM is geographic ({dem_crs}); reprojecting to {proj_fallback}")
            dst_crs = proj_fallback
            transform, width, height = calculate_default_transform(
                src.crs, dst_crs, src.width, src.height, *src.bounds
            )
            kwargs = src.meta.copy()
            kwargs.update({
                "crs": dst_crs,
                "transform": transform,
                "width": width,
                "height": height,
            })
            dem_data = np.empty((height, width), dtype=np.float32)
            reproject(
                source=rio.band(src, 1),
                destination=dem_data,
                src_transform=src.transform,
                src_crs=src.crs,
                dst_transform=transform,
                dst_crs=dst_crs,
                resampling=Resampling.bilinear,
            )
            dem_profile = kwargs
            proj_crs = dst_crs
        else:
            dem_data = src.read(1)
            dem_profile = src.profile
            proj_crs = dem_crs

    return dem_data, dem_profile, proj_crs


def reproject_lonlat_to_xy(lon: np.ndarray, lat: np.ndarray, dst_crs) -> Tuple[np.ndarray, np.ndarray]:
    tf = Transformer.from_crs("EPSG:4326", dst_crs, always_xy=True)
    x, y = tf.transform(lon, lat)
    return np.asarray(x), np.asarray(y)


# =============================================================================
# 3. LOAD DATA
# =============================================================================

def load_surface_dataset(nc_path: Path) -> xr.Dataset:
    print(f"\nLoading surface dataset:\n  {nc_path}")
    ds = xr.open_dataset(nc_path)

    # Force tz-naive time coordinate
    times = pd.to_datetime(ds["time"].values)
    if getattr(times, "tz", None) is not None:
        times = times.tz_localize(None)
    ds = ds.assign_coords(time=times)

    print(ds)
    print("Data variables:", list(ds.data_vars))
    print("Time range:", str(ds["time"].values[0]), "->", str(ds["time"].values[-1]))
    return ds


def load_observations(station_parquet: Path,
                      mros_parquet: Path,
                      start_time: str,
                      end_time: str) -> Tuple[pd.DataFrame, pd.DataFrame]:
    print("\nLoading hourly observations...")
    st = pd.read_parquet(station_parquet)
    mros = pd.read_parquet(mros_parquet)

    st["hour_utc"] = ensure_tz_naive_hour(st["hour_utc"])
    mros["hour_utc"] = ensure_tz_naive_hour(mros["hour_utc"])

    start = pd.to_datetime(start_time)
    end = pd.to_datetime(end_time)

    st = st[(st["hour_utc"] >= start) & (st["hour_utc"] <= end)].copy()
    mros = mros[(mros["hour_utc"] >= start) & (mros["hour_utc"] <= end)].copy()

    # standardize phase names if present
    if "phase" in mros.columns:
        mros["phase"] = mros["phase"].astype(str).str.lower().str.strip()

    print("Stations:", st.shape)
    print("MRoS:", mros.shape)
    return st, mros


# =============================================================================
# 4. LONG FORMAT + SPATIAL SAMPLING
# =============================================================================

def melt_continuous_obs(st: pd.DataFrame) -> pd.DataFrame:
    vars_ = [v for v in ["temp_air", "temp_wet", "temp_dew", "rh"] if v in st.columns]
    out = st.melt(
        id_vars=["hour_utc", "lat", "lon", "elev"],
        value_vars=vars_,
        var_name="var",
        value_name="obs_value",
    ).rename(columns={"hour_utc": "time"})
    out["source"] = "station"
    return out


def build_mros_obs(mros: pd.DataFrame,
                   snow_upper: float,
                   rain_lower: float) -> pd.DataFrame:
    """
    Keep raw MRoS fields for thermodynamic/distribution analysis, and also build
    a long-format table for proxy sampling evaluation.
    """
    required = ["hour_utc", "lat", "lon", "mros_plp_proxy"]
    missing = [c for c in required if c not in mros.columns]
    if missing:
        raise ValueError(f"MRoS parquet is missing required columns: {missing}")

    mros = mros.copy()
    mros["proxy_phase_thresholded"] = classify_mros_proxy(
        mros["mros_plp_proxy"].values,
        snow_upper=snow_upper,
        rain_lower=rain_lower,
    )

    out = mros[["hour_utc", "lat", "lon", "mros_plp_proxy", "elev"]].copy()
    out = out.rename(columns={"hour_utc": "time", "mros_plp_proxy": "obs_value"})
    out["var"] = "mros_plp_proxy"
    out["source"] = "mros"
    return out


def reproject_obs_xy(obs: pd.DataFrame, proj_crs) -> pd.DataFrame:
    obs = obs.copy()
    x, y = reproject_lonlat_to_xy(obs["lon"].values, obs["lat"].values, proj_crs)
    obs["x"] = x
    obs["y"] = y
    return obs


def keep_points_inside_grid(obs: pd.DataFrame, ds: xr.Dataset,
                            xname: str = "x", yname: str = "y") -> pd.DataFrame:
    x_min, x_max = float(ds[xname].min()), float(ds[xname].max())
    y_min, y_max = float(ds[yname].min()), float(ds[yname].max())

    inside = (
        (obs["x"] >= x_min) & (obs["x"] <= x_max) &
        (obs["y"] >= y_min) & (obs["y"] <= y_max)
    )
    pct = 100.0 * inside.mean()
    print(f"{pct:.1f}% of observation points fall inside the surface grid.")
    return obs.loc[inside].copy()


def sample_ds_at_points(ds: xr.Dataset,
                        points_long: pd.DataFrame,
                        var: str,
                        xname: str = "x",
                        yname: str = "y",
                        tname: str = "time") -> np.ndarray:
    """
    Sample gridded surface at observation coordinates/times.
    Uses nearest-time selection, then bilinear spatial interpolation.
    """
    sub = points_long.loc[points_long["var"] == var].copy()
    if sub.empty:
        return np.array([])

    sub["time64"] = sub["time"].values.astype("datetime64[ns]")
    results = []

    for t, g in sub.groupby("time64", sort=True):
        try:
            f_time = ds[var].sel({tname: t}, method="nearest", tolerance=np.timedelta64(30, "m"))
        except Exception:
            try:
                f_time = ds[var].sel({tname: t}, method="nearest")
            except Exception:
                continue

        sampled = f_time.interp(
            {
                xname: xr.DataArray(g[xname].values, dims="obs"),
                yname: xr.DataArray(g[yname].values, dims="obs"),
            }
        )
        results.append(pd.Series(sampled.values, index=g.index))

    if not results:
        return np.array([])

    preds = pd.concat(results).sort_index().values
    return preds


# =============================================================================
# 5. SUPPORT DIAGNOSTICS FOR MRoS
# =============================================================================

def compute_hourly_mros_support(mros: pd.DataFrame, proj_crs) -> pd.DataFrame:
    """
    Quantify how much raw MRoS observational support exists by hour.
    This matters because weak-label defensibility depends strongly on local support.
    """
    if mros.empty:
        return pd.DataFrame()

    df = mros.copy()
    x, y = reproject_lonlat_to_xy(df["lon"].values, df["lat"].values, proj_crs)
    df["x"] = x
    df["y"] = y

    records = []
    for t, g in df.groupby("hour_utc", sort=True):
        rec = {"hour_utc": t, "n_obs": len(g)}

        if len(g) >= 2:
            pts = np.column_stack([g["x"].values, g["y"].values])
            tree = cKDTree(pts)
            dists, _ = tree.query(pts, k=2)
            nn = dists[:, 1]
            rec["mean_nn_dist_m"] = float(np.mean(nn))
            rec["median_nn_dist_m"] = float(np.median(nn))
            rec["max_nn_dist_m"] = float(np.max(nn))
        else:
            rec["mean_nn_dist_m"] = np.nan
            rec["median_nn_dist_m"] = np.nan
            rec["max_nn_dist_m"] = np.nan

        records.append(rec)

    return pd.DataFrame(records).sort_values("hour_utc")


def plot_mros_support(summary: pd.DataFrame, outdir: Path) -> None:
    if summary.empty:
        return

    fig, axes = plt.subplots(1, 2, figsize=(12, 4))

    axes[0].hist(summary["n_obs"], bins=np.arange(0.5, summary["n_obs"].max() + 1.5, 1), edgecolor="black")
    axes[0].set_title("MRoS reports per hour")
    axes[0].set_xlabel("Number of raw MRoS observations")
    axes[0].set_ylabel("Count of hours")

    axes[1].hist(summary["mean_nn_dist_m"].dropna() / 1000.0, bins=25, edgecolor="black")
    axes[1].set_title("Mean nearest-neighbor spacing by hour")
    axes[1].set_xlabel("Mean NN distance (km)")
    axes[1].set_ylabel("Count of hours")

    plt.tight_layout()
    plt.savefig(outdir / "mros_support_summary.png", bbox_inches="tight")
    plt.close()

    summary.to_csv(outdir / "mros_hourly_support_summary.csv", index=False)


# =============================================================================
# 6. CONTINUOUS VARIABLE EVALUATION
# =============================================================================

def evaluate_continuous_vars(ds: xr.Dataset,
                             obs_long: pd.DataFrame,
                             outdir: Path) -> Tuple[pd.DataFrame, pd.DataFrame]:
    vars_ = [v for v in ["temp_air", "temp_wet", "temp_dew", "rh"] if v in ds.data_vars]
    records = []
    residual_frames = []

    for var in vars_:
        sub = obs_long[obs_long["var"] == var].copy()
        if sub.empty:
            continue

        pred = sample_ds_at_points(ds, obs_long, var)
        if len(pred) != len(sub):
            sub = sub.iloc[:len(pred)].copy()

        sub["pred"] = pred
        sub = sub[np.isfinite(sub["obs_value"]) & np.isfinite(sub["pred"])].copy()
        if sub.empty:
            continue

        rec = {
            "var": var,
            "n": len(sub),
            "rmse": rmse(sub["obs_value"].values, sub["pred"].values),
            "mae": float(mean_absolute_error(sub["obs_value"].values, sub["pred"].values)),
            "bias": float(np.mean(sub["pred"].values - sub["obs_value"].values)),
            "corr": safe_corr(sub["obs_value"].values, sub["pred"].values),
            "r2": float(r2_score(sub["obs_value"].values, sub["pred"].values)),
        }
        records.append(rec)

        sub["resid"] = sub["pred"] - sub["obs_value"]
        residual_frames.append(sub)

    metrics = pd.DataFrame(records).sort_values("var")
    residuals = pd.concat(residual_frames, ignore_index=True) if residual_frames else pd.DataFrame()

    metrics.to_csv(outdir / "continuous_interpolation_metrics.csv", index=False)
    residuals.to_csv(outdir / "continuous_point_residuals.csv", index=False)

    return metrics, residuals


def plot_continuous_metrics(metrics: pd.DataFrame, outdir: Path) -> None:
    if metrics.empty:
        return

    fig, axes = plt.subplots(2, 2, figsize=(11, 8))
    axes = axes.ravel()

    sns.barplot(data=metrics, x="var", y="rmse", ax=axes[0], color="steelblue")
    axes[0].set_title("Continuous variables: RMSE")

    sns.barplot(data=metrics, x="var", y="mae", ax=axes[1], color="darkorange")
    axes[1].set_title("Continuous variables: MAE")

    sns.barplot(data=metrics, x="var", y="bias", ax=axes[2], color="gray")
    axes[2].axhline(0, color="black", lw=1)
    axes[2].set_title("Continuous variables: bias")

    sns.barplot(data=metrics, x="var", y="corr", ax=axes[3], color="seagreen")
    axes[3].set_title("Continuous variables: correlation")

    for ax in axes:
        ax.set_xlabel("")
        ax.tick_params(axis="x", rotation=20)

    plt.tight_layout()
    plt.savefig(outdir / "continuous_metrics_summary.png", bbox_inches="tight")
    plt.close()


def plot_continuous_residuals_by_hour(residuals: pd.DataFrame, outdir: Path) -> None:
    if residuals.empty:
        return

    df = residuals.copy()
    df["hour"] = pd.to_datetime(df["time"]).dt.hour

    hourly = (
        df.groupby(["var", "hour"])
          .agg(rmse=("resid", lambda x: np.sqrt(np.nanmean(np.asarray(x) ** 2))),
               bias=("resid", "mean"))
          .reset_index()
    )

    g = sns.FacetGrid(hourly, col="var", col_wrap=2, height=4, sharey=False)
    g.map_dataframe(sns.lineplot, x="hour", y="rmse", marker="o")
    g.set_titles("{col_name}")
    g.set_axis_labels("Hour (UTC)", "RMSE")
    g.savefig(outdir / "continuous_rmse_by_hour.png", bbox_inches="tight")
    plt.close("all")


# =============================================================================
# 7. MROS WEAK-LABEL VALIDATION
# =============================================================================

def evaluate_mros_proxy_continuous(ds: xr.Dataset,
                                   mros_obs_long: pd.DataFrame,
                                   outdir: Path) -> pd.DataFrame:
    """
    Evaluate interpolated MRoS proxy as a continuous weak-label index.
    """
    if "mros_plp_proxy" not in ds.data_vars:
        raise ValueError("Dataset does not contain 'mros_plp_proxy'.")

    sub = mros_obs_long[mros_obs_long["var"] == "mros_plp_proxy"].copy()
    pred = sample_ds_at_points(ds, mros_obs_long, "mros_plp_proxy")

    if len(pred) != len(sub):
        sub = sub.iloc[:len(pred)].copy()

    sub["pred"] = pred
    sub = sub[np.isfinite(sub["obs_value"]) & np.isfinite(sub["pred"])].copy()

    if sub.empty:
        raise ValueError("No valid MRoS proxy samples found.")

    sub["resid"] = sub["pred"] - sub["obs_value"]
    sub["obs_phase_thresholded"] = classify_mros_proxy(sub["obs_value"].values,
                                                       CONFIG["mros_snow_upper"],
                                                       CONFIG["mros_rain_lower"])
    sub["pred_phase_thresholded"] = classify_mros_proxy(sub["pred"].values,
                                                        CONFIG["mros_snow_upper"],
                                                        CONFIG["mros_rain_lower"])

    metrics = pd.DataFrame([{
        "var": "mros_plp_proxy",
        "n": len(sub),
        "rmse": rmse(sub["obs_value"].values, sub["pred"].values),
        "mae": float(mean_absolute_error(sub["obs_value"].values, sub["pred"].values)),
        "bias": float(np.mean(sub["pred"].values - sub["obs_value"].values)),
        "corr": safe_corr(sub["obs_value"].values, sub["pred"].values),
        "r2": float(r2_score(sub["obs_value"].values, sub["pred"].values)),
    }])

    metrics.to_csv(outdir / "mros_proxy_continuous_metrics.csv", index=False)
    sub.to_csv(outdir / "mros_proxy_sampled_points.csv", index=False)

    return sub


def evaluate_mros_phase_recovery(mros_eval_df: pd.DataFrame,
                                 snow_upper: float,
                                 rain_lower: float,
                                 out_csv: Optional[Path] = None) -> pd.DataFrame:
    """
    Evaluate how well sampled/interpolated MRoS proxy recovers thresholded phase categories.
    """
    df = mros_eval_df.copy()

    df["obs_phase"] = classify_mros_proxy(df["obs_value"].values, snow_upper, rain_lower)
    df["pred_phase"] = classify_mros_proxy(df["pred"].values, snow_upper, rain_lower)

    valid = df["obs_phase"].notna() & df["pred_phase"].notna()
    df = df.loc[valid].copy()

    labels = ordered_phase_labels()
    cm = confusion_matrix(df["obs_phase"], df["pred_phase"], labels=labels)
    bal_acc = balanced_accuracy_score(df["obs_phase"], df["pred_phase"])
    precision, recall, f1, support = precision_recall_fscore_support(
        df["obs_phase"], df["pred_phase"], labels=labels, zero_division=0
    )

    rows = []
    rows.append({
        "metric": "balanced_accuracy",
        "class": "overall",
        "value": float(bal_acc),
        "snow_upper": snow_upper,
        "rain_lower": rain_lower,
    })

    for i, lab in enumerate(labels):
        rows.append({
            "metric": "precision",
            "class": lab,
            "value": float(precision[i]),
            "snow_upper": snow_upper,
            "rain_lower": rain_lower,
        })
        rows.append({
            "metric": "recall",
            "class": lab,
            "value": float(recall[i]),
            "snow_upper": snow_upper,
            "rain_lower": rain_lower,
        })
        rows.append({
            "metric": "f1",
            "class": lab,
            "value": float(f1[i]),
            "snow_upper": snow_upper,
            "rain_lower": rain_lower,
        })
        rows.append({
            "metric": "support",
            "class": lab,
            "value": int(support[i]),
            "snow_upper": snow_upper,
            "rain_lower": rain_lower,
        })

    out = pd.DataFrame(rows)

    if out_csv is not None:
        out.to_csv(out_csv, index=False)

    return out


def run_mros_threshold_sensitivity(mros_eval_df: pd.DataFrame,
                                   snow_upper_grid: Iterable[float],
                                   rain_lower_grid: Iterable[float],
                                   outdir: Path) -> pd.DataFrame:
    """
    Explore how phase-recovery performance depends on threshold choices.
    """
    records = []

    for s_up in snow_upper_grid:
        for r_lo in rain_lower_grid:
            if s_up >= r_lo:
                continue

            df = mros_eval_df.copy()
            df["obs_phase"] = classify_mros_proxy(df["obs_value"].values, s_up, r_lo)
            df["pred_phase"] = classify_mros_proxy(df["pred"].values, s_up, r_lo)

            valid = df["obs_phase"].notna() & df["pred_phase"].notna()
            df = df.loc[valid].copy()
            if df.empty:
                continue

            labels = ordered_phase_labels()

            try:
                bal_acc = balanced_accuracy_score(df["obs_phase"], df["pred_phase"])
                precision, recall, f1, support = precision_recall_fscore_support(
                    df["obs_phase"], df["pred_phase"], labels=labels, zero_division=0
                )
            except Exception:
                continue

            records.append({
                "snow_upper": s_up,
                "rain_lower": r_lo,
                "balanced_accuracy": float(bal_acc),
                "snow_f1": float(f1[0]),
                "mix_f1": float(f1[1]),
                "rain_f1": float(f1[2]),
                "snow_recall": float(recall[0]),
                "mix_recall": float(recall[1]),
                "rain_recall": float(recall[2]),
                "n": len(df),
            })

    out = pd.DataFrame(records).sort_values(["balanced_accuracy", "mix_f1"], ascending=False)
    out.to_csv(outdir / "mros_threshold_sensitivity.csv", index=False)
    return out


# =============================================================================
# 8. MROS DISTRIBUTION / DIAGNOSTIC PLOTS
# =============================================================================

def attach_station_thermo_to_mros(mros: pd.DataFrame, st: pd.DataFrame) -> pd.DataFrame:
    """
    Attach nearest station-hour thermodynamic variables to each raw MRoS observation.
    This supports foundational raw phase-vs-temperature distribution plots.

    Match rule:
      - same hour
      - nearest station in projected space
    """
    if mros.empty or st.empty:
        return mros.copy()

    st_use = st.copy()
    mros_use = mros.copy()

    # Need projected x/y. Re-use DEM CRS from surface/DEM later outside.
    return mros_use


def attach_nearest_station_thermo_to_mros(mros: pd.DataFrame,
                                          st: pd.DataFrame,
                                          proj_crs) -> pd.DataFrame:
    """
    For each raw MRoS observation, find nearest station at the same hour and copy over:
      temp_air, temp_wet, temp_dew, rh
    """
    if mros.empty or st.empty:
        return mros.copy()

    mros = mros.copy()
    st = st.copy()

    mx, my = reproject_lonlat_to_xy(mros["lon"].values, mros["lat"].values, proj_crs)
    sx, sy = reproject_lonlat_to_xy(st["lon"].values, st["lat"].values, proj_crs)
    mros["x"] = mx
    mros["y"] = my
    st["x"] = sx
    st["y"] = sy

    cols_keep = ["hour_utc", "x", "y", "temp_air", "temp_wet", "temp_dew", "rh"]
    st = st[cols_keep].copy()

    out_frames = []

    for t, gm in mros.groupby("hour_utc", sort=True):
        gs = st[st["hour_utc"] == t].copy()
        if gs.empty:
            gcopy = gm.copy()
            for col in ["temp_air", "temp_wet", "temp_dew", "rh", "nearest_station_dist_m"]:
                gcopy[col] = np.nan
            out_frames.append(gcopy)
            continue

        tree = cKDTree(np.column_stack([gs["x"].values, gs["y"].values]))
        dists, idx = tree.query(np.column_stack([gm["x"].values, gm["y"].values]), k=1)

        matched = gs.iloc[idx].reset_index(drop=True)
        gcopy = gm.reset_index(drop=True).copy()
        for col in ["temp_air", "temp_wet", "temp_dew", "rh"]:
            gcopy[col] = matched[col].values
        gcopy["nearest_station_dist_m"] = dists
        out_frames.append(gcopy)

    return pd.concat(out_frames, ignore_index=True)


def plot_raw_mros_kdes(mros_with_thermo: pd.DataFrame, outdir: Path) -> None:
    """
    Kernel densities of raw MRoS phases against air temp and wet-bulb temp.
    These plots are foundational because they use raw phase observations first.
    """
    df = mros_with_thermo.copy()
    if "phase" not in df.columns:
        return

    phases = ["snow", "mix", "rain"]
    colors = {"snow": "#3b6fb6", "mix": "#c03a98", "rain": "#3a9c5d"}

    for temp_var in ["temp_air", "temp_wet"]:
        if temp_var not in df.columns:
            continue

        sub = df[df["phase"].isin(phases) & df[temp_var].notna()].copy()
        if sub.empty:
            continue

        plt.figure(figsize=(8, 5))
        for ph in phases:
            d = sub.loc[sub["phase"] == ph, temp_var].dropna()
            if len(d) < 5:
                continue
            sns.kdeplot(d, label=ph, fill=False, lw=2, color=colors[ph])

        plt.axvline(0, color="black", ls="--", lw=1)
        plt.title(f"Raw MRoS phase distributions vs {temp_var}")
        plt.xlabel(f"{temp_var} (°C)")
        plt.ylabel("Density")
        plt.legend()
        plt.tight_layout()
        plt.savefig(outdir / f"raw_mros_kde_{temp_var}.png", bbox_inches="tight")
        plt.close()


def plot_mros_proxy_residuals(mros_eval_df: pd.DataFrame, outdir: Path) -> None:
    df = mros_eval_df.copy()

    fig, axes = plt.subplots(1, 3, figsize=(15, 4.3))

    sns.histplot(df["resid"], bins=40, kde=True, ax=axes[0], color="gray")
    axes[0].axvline(0, color="black", lw=1)
    axes[0].set_title("MRoS proxy residuals")
    axes[0].set_xlabel("Pred - Obs proxy")

    axes[1].scatter(df["obs_value"], df["pred"], s=10, alpha=0.4)
    axes[1].plot([0, 100], [0, 100], color="black", ls="--", lw=1)
    axes[1].set_title("MRoS proxy: observed vs sampled")
    axes[1].set_xlabel("Observed proxy")
    axes[1].set_ylabel("Sampled grid proxy")

    by_phase = df.copy()
    sns.boxplot(data=by_phase, x="obs_phase_thresholded", y="resid",
                order=ordered_phase_labels(), ax=axes[2])
    axes[2].axhline(0, color="black", lw=1)
    axes[2].set_title("Residuals by thresholded observed phase")
    axes[2].set_xlabel("Observed thresholded phase")
    axes[2].set_ylabel("Pred - Obs proxy")

    plt.tight_layout()
    plt.savefig(outdir / "mros_proxy_residual_diagnostics.png", bbox_inches="tight")
    plt.close()


def plot_mros_confusion_matrix(mros_eval_df: pd.DataFrame,
                               snow_upper: float,
                               rain_lower: float,
                               outdir: Path) -> None:
    df = mros_eval_df.copy()
    df["obs_phase"] = classify_mros_proxy(df["obs_value"].values, snow_upper, rain_lower)
    df["pred_phase"] = classify_mros_proxy(df["pred"].values, snow_upper, rain_lower)

    labels = ordered_phase_labels()
    cm = confusion_matrix(df["obs_phase"], df["pred_phase"], labels=labels)

    plt.figure(figsize=(5.5, 4.8))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
                xticklabels=labels, yticklabels=labels)
    plt.title(f"MRoS phase recovery confusion matrix\n(snow <= {snow_upper}, rain >= {rain_lower})")
    plt.xlabel("Predicted phase")
    plt.ylabel("Observed phase")
    plt.tight_layout()
    plt.savefig(outdir / "mros_phase_confusion_matrix.png", bbox_inches="tight")
    plt.close()


def plot_mros_threshold_heatmaps(sensitivity_df: pd.DataFrame, outdir: Path) -> None:
    if sensitivity_df.empty:
        return

    for metric in ["balanced_accuracy", "mix_f1", "snow_f1", "rain_f1", "mix_recall"]:
        if metric not in sensitivity_df.columns:
            continue

        piv = sensitivity_df.pivot(index="snow_upper", columns="rain_lower", values=metric)

        plt.figure(figsize=(6, 5))
        sns.heatmap(piv, annot=True, fmt=".2f", cmap="viridis")
        plt.title(f"MRoS threshold sensitivity: {metric}")
        plt.xlabel("rain_lower")
        plt.ylabel("snow_upper")
        plt.tight_layout()
        plt.savefig(outdir / f"mros_threshold_heatmap_{metric}.png", bbox_inches="tight")
        plt.close()


def build_phase_fraction_table(df: pd.DataFrame,
                               temp_col: str,
                               phase_col: str,
                               group_name: str,
                               bins: np.ndarray) -> pd.DataFrame:
    sub = df[[temp_col, phase_col]].dropna().copy()
    if sub.empty:
        return pd.DataFrame()

    sub["temp_bin"] = pd.cut(sub[temp_col], bins=bins, include_lowest=True)
    counts = sub.groupby(["temp_bin", phase_col]).size().rename("n").reset_index()
    total = counts.groupby("temp_bin")["n"].sum().rename("n_total").reset_index()
    counts = counts.merge(total, on="temp_bin", how="left")
    counts["fraction"] = counts["n"] / counts["n_total"]
    counts["group"] = group_name
    counts["temp_mid"] = counts["temp_bin"].apply(lambda x: x.mid if pd.notna(x) else np.nan)
    return counts


def plot_phase_fraction_by_temp_bin(raw_mros_df: pd.DataFrame,
                                    mros_eval_df: pd.DataFrame,
                                    temp_var: str,
                                    outdir: Path,
                                    bins: np.ndarray) -> None:
    """
    Compare phase fractions across temperature bins for:
      1) raw MRoS observed phase
      2) sampled interpolated weak-label phase
    """
    tables = []

    # raw MRoS observed phase
    if ("phase" in raw_mros_df.columns) and (temp_var in raw_mros_df.columns):
        t1 = build_phase_fraction_table(raw_mros_df, temp_var, "phase", "raw_mros_obs", bins)
        tables.append(t1)

    # sampled interpolated phase at observed MRoS points
    if temp_var in mros_eval_df.columns:
        tmp = mros_eval_df.copy()
        tmp["pred_phase"] = classify_mros_proxy(tmp["pred"].values,
                                                CONFIG["mros_snow_upper"],
                                                CONFIG["mros_rain_lower"])
        t2 = build_phase_fraction_table(tmp, temp_var, "pred_phase", "sampled_surface_phase", bins)
        tables.append(t2)

    if not tables:
        return

    plot_df = pd.concat([t for t in tables if not t.empty], ignore_index=True)
    if plot_df.empty:
        return

    phases = ordered_phase_labels()
    fig, axes = plt.subplots(1, len(plot_df["group"].unique()), figsize=(6 * len(plot_df["group"].unique()), 4.5), sharey=True)
    if not isinstance(axes, np.ndarray):
        axes = np.array([axes])

    for ax, grp in zip(axes, plot_df["group"].unique()):
        sub = plot_df[plot_df["group"] == grp]
        for ph in phases:
            d = sub[sub.iloc[:, 1] == ph]
            if d.empty:
                continue
            ax.plot(d["temp_mid"], d["fraction"], marker="o", label=ph)

        ax.axvline(0, color="black", ls="--", lw=1)
        ax.set_title(f"{grp}\nPhase fraction vs {temp_var}")
        ax.set_xlabel(f"{temp_var} (°C)")
        ax.set_ylabel("Fraction")
        ax.set_ylim(0, 1)

    axes[0].legend()
    plt.tight_layout()
    plt.savefig(outdir / f"phase_fraction_by_{temp_var}.png", bbox_inches="tight")
    plt.close()


# =============================================================================
# 9. MAIN
# =============================================================================

def main():
    print("=" * 80)
    print("WEAK-LABEL VALIDATION + INTERPOLATION EVALUATION")
    print("=" * 80)

    # -------------------------------------------------------------------------
    # Load DEM / projection
    # -------------------------------------------------------------------------
    dem_data, dem_profile, proj_crs = load_dem_and_projected_profile(
        CONFIG["dem_path"], CONFIG["proj_fallback"]
    )
    print(f"Using projected CRS: {proj_crs}")

    # -------------------------------------------------------------------------
    # Load gridded surface + observations
    # -------------------------------------------------------------------------
    ds = load_surface_dataset(CONFIG["surface_nc"])
    st, mros = load_observations(
        CONFIG["station_hourly_parquet"],
        CONFIG["mros_hourly_parquet"],
        CONFIG["start_time"],
        CONFIG["end_time"],
    )

    # -------------------------------------------------------------------------
    # MRoS support diagnostics
    # -------------------------------------------------------------------------
    print("\nComputing MRoS support diagnostics...")
    mros_support = compute_hourly_mros_support(mros, proj_crs)
    if not mros_support.empty:
        print(mros_support.describe(include="all"))
    plot_mros_support(mros_support, OUT_DIR)

    # -------------------------------------------------------------------------
    # Build long-format tables
    # -------------------------------------------------------------------------
    cont_long = melt_continuous_obs(st)
    mros_long = build_mros_obs(
        mros,
        snow_upper=CONFIG["mros_snow_upper"],
        rain_lower=CONFIG["mros_rain_lower"],
    )

    obs_all = pd.concat([cont_long, mros_long], ignore_index=True)
    obs_all["time"] = ensure_tz_naive_hour(obs_all["time"])

    # Project point coords
    obs_all = reproject_obs_xy(obs_all, proj_crs)
    obs_all = keep_points_inside_grid(obs_all, ds)

    # Keep split tables too
    cont_long = obs_all[obs_all["source"] == "station"].copy()
    mros_long = obs_all[obs_all["source"] == "mros"].copy()

    # -------------------------------------------------------------------------
    # Continuous-variable interpolation QA/QC
    # -------------------------------------------------------------------------
    print("\nEvaluating continuous variables...")
    cont_metrics, cont_resid = evaluate_continuous_vars(ds, cont_long, OUT_DIR)
    print("\nContinuous metrics:")
    print(cont_metrics)

    plot_continuous_metrics(cont_metrics, OUT_DIR)
    plot_continuous_residuals_by_hour(cont_resid, OUT_DIR)

    # -------------------------------------------------------------------------
    # Attach nearest station thermo to raw MRoS for foundational distributions
    # -------------------------------------------------------------------------
    print("\nAttaching nearest station thermodynamics to raw MRoS observations...")
    raw_mros_with_thermo = attach_nearest_station_thermo_to_mros(mros, st, proj_crs)
    raw_mros_with_thermo.to_csv(OUT_DIR / "raw_mros_with_nearest_station_thermo.csv", index=False)

    plot_raw_mros_kdes(raw_mros_with_thermo, OUT_DIR)

    # -------------------------------------------------------------------------
    # MRoS continuous-index evaluation
    # -------------------------------------------------------------------------
    print("\nEvaluating MRoS weak-label proxy as continuous index...")
    mros_eval_df = evaluate_mros_proxy_continuous(ds, mros_long, OUT_DIR)
    print(mros_eval_df[["obs_value", "pred", "resid"]].describe())

    # Attach nearest station thermo to sampled MRoS evaluation rows too
    thermo_cols = ["hour_utc", "lat", "lon", "phase", "temp_air", "temp_wet", "temp_dew", "rh", "nearest_station_dist_m"]
    keep_cols = [c for c in thermo_cols if c in raw_mros_with_thermo.columns]
    merge_df = raw_mros_with_thermo[keep_cols].copy()

    mros_eval_df = mros_eval_df.rename(columns={"time": "hour_utc"})
    mros_eval_df = mros_eval_df.merge(
        merge_df,
        on=["hour_utc", "lat", "lon"],
        how="left",
        suffixes=("", "_raw"),
    )
    mros_eval_df.to_csv(OUT_DIR / "mros_proxy_sampled_points_enriched.csv", index=False)

    plot_mros_proxy_residuals(mros_eval_df, OUT_DIR)

    # -------------------------------------------------------------------------
    # MRoS phase-recovery at default thresholds
    # -------------------------------------------------------------------------
    print("\nEvaluating MRoS phase recovery at default thresholds...")
    phase_metrics = evaluate_mros_phase_recovery(
        mros_eval_df.rename(columns={"hour_utc": "time"}),
        snow_upper=CONFIG["mros_snow_upper"],
        rain_lower=CONFIG["mros_rain_lower"],
        out_csv=OUT_DIR / "mros_phase_recovery_metrics_default_thresholds.csv",
    )
    print(phase_metrics)

    plot_mros_confusion_matrix(
        mros_eval_df.rename(columns={"hour_utc": "time"}),
        snow_upper=CONFIG["mros_snow_upper"],
        rain_lower=CONFIG["mros_rain_lower"],
        outdir=OUT_DIR,
    )

    # -------------------------------------------------------------------------
    # Threshold sensitivity
    # -------------------------------------------------------------------------
    print("\nRunning MRoS threshold sensitivity analysis...")
    sensitivity = run_mros_threshold_sensitivity(
        mros_eval_df.rename(columns={"hour_utc": "time"}),
        snow_upper_grid=CONFIG["snow_upper_grid"],
        rain_lower_grid=CONFIG["rain_lower_grid"],
        outdir=OUT_DIR,
    )
    print("\nTop threshold combinations:")
    print(sensitivity.head(10))

    plot_mros_threshold_heatmaps(sensitivity, OUT_DIR)

    # -------------------------------------------------------------------------
    # Phase-fraction-by-temperature diagnostics
    # -------------------------------------------------------------------------
    print("\nPlotting phase fractions by temperature bins...")
    bins = make_temp_bins(
        CONFIG["temp_bin_min"],
        CONFIG["temp_bin_max"],
        CONFIG["temp_bin_width"],
    )

    for temp_var in ["temp_air", "temp_wet"]:
        plot_phase_fraction_by_temp_bin(
            raw_mros_df=raw_mros_with_thermo,
            mros_eval_df=mros_eval_df,
            temp_var=temp_var,
            outdir=OUT_DIR,
            bins=bins,
        )

    # -------------------------------------------------------------------------
    # Final summary
    # -------------------------------------------------------------------------
    summary_lines = [
        "Weak-label validation completed.",
        f"Surface file evaluated: {CONFIG['surface_nc']}",
        f"Output directory: {OUT_DIR}",
        "",
        "Core outputs to inspect first:",
        "  1) mros_support_summary.png",
        "  2) raw_mros_kde_temp_wet.png",
        "  3) mros_proxy_continuous_metrics.csv",
        "  4) mros_phase_confusion_matrix.png",
        "  5) mros_threshold_heatmap_mix_f1.png",
        "  6) phase_fraction_by_temp_wet.png",
    ]
    summary_txt = "\n".join(summary_lines)

    with open(OUT_DIR / "README_summary.txt", "w", encoding="utf-8") as f:
        f.write(summary_txt)

    print("\n" + "=" * 80)
    print(summary_txt)
    print("=" * 80)

    ds.close()


if __name__ == "__main__":
    main()


WEAK-LABEL VALIDATION + INTERPOLATION EVALUATION
Using projected CRS: EPSG:26911

Loading surface dataset:
  C:\Users\EmmaGolub\Desktop\MRoS_local\local_data\IDW\hourly_predictors_1km_IDW_full.nc
<xarray.Dataset> Size: 5GB
Dimensions:         (time: 5832, y: 251, x: 142)
Coordinates:
  * y               (y) float64 2kB 4.426e+06 4.425e+06 ... 4.177e+06 4.176e+06
  * x               (x) float64 1kB 1.626e+05 1.636e+05 ... 3.026e+05 3.036e+05
  * time            (time) datetime64[ns] 47kB 2024-10-01 ... 2025-05-31T23:0...
Data variables:
    temp_air        (time, y, x) float32 831MB ...
    temp_dew        (time, y, x) float32 831MB ...
    temp_wet        (time, y, x) float32 831MB ...
    rh              (time, y, x) float32 831MB ...
    mros_plp_proxy  (time, y, x) float32 831MB ...
    plp             (time, y, x) float32 831MB ...
    elev            (y, x) float32 143kB ...
    spatial_ref     int64 8B ...
Attributes:
    title:             Hourly predictor stacks on 1-km grid
  